# First-contact emitter assignment with the emitter-aware ontology

This notebook ingests the first `PY_CONTACT_LOG` line from `LuaHistory_2026-06-23.txt`, extracts CMO observation features, retrieves platform/operator hypotheses from the new KG ontology, applies the probability layer, and builds an LLM explanation payload.

In [ ]:
from pathlib import Path
import os, sys, json, math
from dataclasses import asdict

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'combat_id_calibration').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

LUA_HISTORY = REPO_ROOT / 'LuaHistory_2026-06-23.txt'
WORK_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'first_contact_emitter_assignment'
WORK_DIR.mkdir(parents=True, exist_ok=True)

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or None

In [ ]:
from combat_id_calibration.cmo_observation_ingest import parse_observation_line, write_observations_jsonl, populate_observations_neo4j
first_line = next(line for line in LUA_HISTORY.read_text(encoding='utf-8-sig', errors='replace').splitlines() if line.startswith('PY_CONTACT_LOG'))
obs = parse_observation_line(first_line, source_line=1)
write_observations_jsonl([obs], WORK_DIR / 'first_contact_observation.jsonl')
asdict(obs)

In [ ]:
# Optional: write the dynamic observation into the same Neo4j graph.
if NEO4J_PASSWORD:
    populate_observations_neo4j([obs], NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
    print('First observation ingested into Neo4j')
else:
    print('Set NEO4J_PASSWORD to ingest the observation into Neo4j')

In [ ]:
from combat_id_calibration.hypothesis_generation import fetch_graph_hypotheses, select_offline_hypotheses, emitter_aliases

seed_candidates = [
    {'hypothesis':'MiG-29 Fulcrum C', 'operator_nation':'Ukraine', 'emitter_aliases':['Slot Back [N-010 Zhuk-M]','N-010 Zhuk-M','Zhuk-M'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,800], 'typical_altitude_m':[0,18000], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
    {'hypothesis':'MiG-29SMT', 'operator_nation':'Russia', 'emitter_aliases':['N-010 Zhuk-M','Zhuk-M'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,800], 'typical_altitude_m':[0,18000], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
    {'hypothesis':'MiG-35', 'operator_nation':'Russia', 'emitter_aliases':['Zhuk-M','Zhuk-AE'], 'platform_class':obs.emission_target_type, 'typical_speed_kt':[250,900], 'typical_altitude_m':[0,17500], 'kg_support_count':1, 'evidence_paths':['offline_seed']},
]
if NEO4J_PASSWORD:
    kg_rows = fetch_graph_hypotheses(obs, 10, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE) or seed_candidates
else:
    kg_rows = seed_candidates
hypotheses = select_offline_hypotheses(obs, kg_rows, 5)
hypotheses

In [ ]:
from combat_id_calibration.probability_model import platform_operator_nation_distribution
from combat_id_calibration.hypothesis_generation import evidence_paths_query_id

SCENARIO_ID = 'LuaHistory_2026-06-23_first_contact'

def hypothesis_feature_row(obs, h):
    alias_match = any(a.lower() in obs.emission_sensor_name.lower() for a in h.get('emitter_aliases', []))
    speed_low, speed_high = h.get('typical_speed_kt', [0, 2500])
    alt_low, alt_high = h.get('typical_altitude_m', [0, 25000])
    speed_match = speed_low <= obs.emission_speed <= speed_high
    altitude_match = alt_low <= obs.emission_altitude <= alt_high
    support_count = float(h.get('kg_support_count', len(h.get('evidence_paths', [])) or 0))
    # These feature columns are the contract consumed by probability_model.py.
    return {
        'scenario_id': SCENARIO_ID,
        'contact_id': obs.observation_id,
        'observation_time': obs.time,
        'hypothesis': h['hypothesis'],
        'operator_nation': h.get('operator_nation', 'unknown'),
        'supporting_path_count': support_count,
        'contradicting_path_count': float(h.get('contradicting_path_count', 0)),
        'mean_source_reliability': float(h.get('mean_source_reliability', 0.75)),
        'recency': float(h.get('recency', 1.0)),
        'shortest_path_to_platform_class': float(h.get('shortest_path_to_platform_class', 1 if h.get('platform_class') else 0)),
        'emission_match_score': 1.0 if alias_match else 0.0,
        'kinematic_match_score': (float(speed_match) + float(altitude_match)) / 2.0,
        'contradiction_score': float(h.get('contradiction_score', 0.0)),
        'evidence_query_id': h.get('evidence_query_id') or evidence_paths_query_id(h.get('evidence_paths', [])),
        'features': {
            'emitter_alias_match': alias_match,
            'observed_speed_kt': obs.emission_speed,
            'observed_altitude_m': obs.emission_altitude,
            'observed_latitude': obs.emission_latitude,
            'observed_longitude': obs.emission_longitude,
        },
    }

feature_rows = [hypothesis_feature_row(obs, h) for h in hypotheses]
probability_record = platform_operator_nation_distribution(feature_rows)
assignments = [
    {
        **next(h for h in hypotheses if h['hypothesis'] == candidate['platform']),
        'probability': candidate['probability'],
        'operator_nation': candidate['operator_nation'],
        'logit': candidate['logit'],
        'evidence_query_id': candidate['evidence_query_id'],
        'features': next(row['features'] for row in feature_rows if row['hypothesis'] == candidate['platform']),
    }
    for candidate in probability_record['candidates']
]
(WORK_DIR / 'first_contact_feature_rows.jsonl').write_text(''.join(json.dumps(row, sort_keys=True)+'\n' for row in feature_rows), encoding='utf-8')
(WORK_DIR / 'first_contact_probability_assignment.jsonl').write_text(json.dumps(probability_record, sort_keys=True)+'\n', encoding='utf-8')
probability_record



In [ ]:
from combat_id_calibration.hypothesis_generation import build_llm_hypothesis_prompt
from combat_id_calibration.llm_explainer import build_explanation_payload

evidence = {
    'supporting_evidence': [
        {
            'text': f"{row['hypothesis']} has emitter_alias_match={row['features']['emitter_alias_match']} and kinematic_match_score={row['kinematic_match_score']:.2f}",
            'source': row.get('evidence_query_id') or row['hypothesis'],
        }
        for row in feature_rows
    ],
    'missing_evidence': [
        'Collect additional emitter scans, track kinematics, IFF, location context, and source reliability before treating this as definitive.'
    ],
}
explanation_payload = build_explanation_payload(probability_record, evidence)
explanation_payload.update({
    'observation': asdict(obs),
    'emitter_aliases': emitter_aliases(obs.emission_sensor_name),
    'probability_assignments': assignments,
    'llm_instruction': 'Explain why the probability model favored these identities/operators. Do not change probabilities.',
    'hypothesis_prompt_context': build_llm_hypothesis_prompt(obs, hypotheses, len(hypotheses)),
})
(WORK_DIR / 'first_contact_explanation_payload.json').write_text(json.dumps(explanation_payload, indent=2, sort_keys=True), encoding='utf-8')
explanation_payload



In [ ]:
import re

def _extract_json_object(text):
    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

def explain_with_llm(explanation_payload, model=None):
    """Send the explanation payload to an LLM and return text plus model probability.

    Requires `pip install openai` and `OPENAI_API_KEY`. The LLM is downstream of
    probability_model.py: it explains the supplied probability, but never computes
    or edits the probability. If no API key is configured, the deterministic
    payload summary is returned so the notebook remains runnable offline.
    """
    top_probability = float(probability_record['top_platform_probability'])
    if not os.getenv('OPENAI_API_KEY'):
        return {
            'probability': top_probability,
            'explanation': explanation_payload['summary'],
            'llm_raw_output': None,
            'note': 'Set OPENAI_API_KEY to call an LLM; returned deterministic payload summary instead.',
        }

    from openai import OpenAI

    client = OpenAI()
    response = client.responses.create(
        model=model or os.getenv('OPENAI_MODEL', 'gpt-4.1-mini'),
        input=[
            {
                'role': 'system',
                'content': (
                    'You explain calibrated combat-identification model outputs. '
                    'Do not alter, recalculate, round aggressively, or invent probabilities. '
                    'Return JSON with keys explanation and probability.'
                ),
            },
            {
                'role': 'user',
                'content': json.dumps(explanation_payload, indent=2, sort_keys=True),
            },
        ],
    )
    raw_text = response.output_text
    parsed = _extract_json_object(raw_text) or {}
    return {
        'probability': top_probability,
        'explanation': parsed.get('explanation', raw_text),
        'llm_raw_output': raw_text,
    }

llm_explanation = explain_with_llm(explanation_payload)
(WORK_DIR / 'first_contact_llm_explanation.json').write_text(json.dumps(llm_explanation, indent=2, sort_keys=True), encoding='utf-8')
llm_explanation

